Running IsoVita pipeline on hippocampus data - figures are generated in another notebook

**Data loading and Preprocessing**

In [5]:
from pathlib import Path

import anndata as ad
import pandas as pd

from prepare_isovita_inputs import (
    prepare_isovita_inputs,
    make_underscore_prefix_parser,
    save_prepared_inputs,
)


# Paths
DATA_DIR = Path("/users/xchen5/IsoVita/data/HPC")

H5AD_PATH = DATA_DIR / "HPC_transcript_counts.h5ad"
METADATA_PATH = DATA_DIR / "metadata.tsv"
TX2GENE_PATH = DATA_DIR / "tx2gene.csv"
OUTPUT_DIR = DATA_DIR / "isovita_inputs"


# Load inputs
adata = ad.read_h5ad(H5AD_PATH)
metadata = pd.read_csv(METADATA_PATH, sep="\t")
tx2gene = pd.read_csv(TX2GENE_PATH)


# Parse AnnData sample names to metadata sample IDs
sample_id_parser = make_underscore_prefix_parser(n_parts=2)


# Prepare canonical IsoVita inputs
prepared = prepare_isovita_inputs(
    adata=adata,
    metadata=metadata,
    tx2gene=tx2gene,
    metadata_sample_col="sample_id",
    age_col="age",
    metadata_filters={"region": "HIPPO"},
    abundance_key="abundance",
    sample_id_parser=sample_id_parser,
    tx2gene_transcript_col="transcript_id",
    tx2gene_gene_col="gene_id",
    postnatal_only=True,
)


# Final alignment check
tpm_sample_cols = [
    c
    for c in prepared.tpm.columns
    if c not in {"gene_id", "transcript_id"}
]

assert tpm_sample_cols == prepared.metadata["sample_id"].astype(str).tolist()


# Save canonical inputs
save_prepared_inputs(
    prepared,
    OUTPUT_DIR,
)


# Preparation summary
print(prepared.summary.to_string(index=False))
print(f"\nSaved to: {OUTPUT_DIR}")

              metric     value
    retained_samples    442.00
retained_transcripts 157458.00
      retained_genes  57960.00
             age_min      0.07
             age_max     96.92
      n_negative_age      0.00

Saved to: /users/xchen5/IsoVita/data/HPC/isovita_inputs


**Running `IsoVita` pipeline**

In [2]:
# ============================================================
# Running IsoVita pipeline
# ============================================================

from pathlib import Path
import shutil

import pandas as pd

from isovita import (
    IsoVitaConfig,
    PermutationInferenceConfig,
    run_isovita_pipeline,
    save_isovita_pipeline_outputs,
)


# ============================================================
# Paths
# ============================================================

DATA_DIR = Path(
    "/users/xchen5/IsoVita/data/HPC"
)

INPUT_DIR = (
    DATA_DIR
    / "isovita_inputs"
)

OUTPUT_DIR = (
    DATA_DIR
    / "isovita_results"
)


# ============================================================
# Load prepared inputs
# ============================================================

tpm = pd.read_csv(
    INPUT_DIR
    / "tpm.tsv",
    sep="\t",
)

metadata = pd.read_csv(
    INPUT_DIR
    / "metadata.tsv",
    sep="\t",
)


# ============================================================
# IsoVita configuration
# ============================================================

config = IsoVitaConfig(
    inference=PermutationInferenceConfig(
        n_permutations=5000,
        random_seed=0,
        batch_size=128,
    ),
)


# ============================================================
# Run production IsoVita
# ============================================================

result = run_isovita_pipeline(
    tpm=tpm,
    metadata=metadata,
    config=config,
    progress_every=1000, # progress report
    n_jobs=16, # shared-memory worker threads for permutation inference.
)


# ============================================================
# Save outputs
# ============================================================

if OUTPUT_DIR.exists():
    shutil.rmtree(
        OUTPUT_DIR
    )


save_isovita_pipeline_outputs(
    result,
    OUTPUT_DIR,
)


# ============================================================
# Run summary
# ============================================================

print(
    "\nIsoVita run complete"
)

print(
    f"Output directory: "
    f"{OUTPUT_DIR}"
)


print(
    f"\nGenes entering balance construction: "
    f"{(
        result.gene_routes[
            'route'
        ]
        == 'balance_ge2'
    ).sum():,}"
)


print(
    f"Successful balance features: "
    f"{(
        result.balance_summary[
            'status'
        ]
        == 'ok'
    ).sum():,}"
)


print(
    f"Discrete discoveries: "
    f"{len(
        result.discrete_trajectory_candidates
    ):,}"
)


print(
    f"Continuous discoveries: "
    f"{len(
        result.continuous_trajectory_candidates
    ):,}"
)


print(
    f"Unique genes in production-branch union: "
    f"{len(
        result.final_features
    ):,}"
)

Balance: 0/9371
Balance: 1000/9371
Balance: 2000/9371
Balance: 3000/9371
Balance: 4000/9371
Balance: 5000/9371
Balance: 6000/9371
Balance: 7000/9371
Balance: 8000/9371
Balance: 9000/9371
Selection-aware inference: 0/9371
Selection-aware inference: 1000/9371
Selection-aware inference: 2000/9371
Selection-aware inference: 3000/9371
Selection-aware inference: 4000/9371
Selection-aware inference: 5000/9371
Selection-aware inference: 6000/9371
Selection-aware inference: 7000/9371
Selection-aware inference: 8000/9371
Selection-aware inference: 9000/9371


/users/xchen5/IsoVita/isovita.py:2787: RuntimeWarning: Discrete permutation resolution may be coarse for BH: minimum p=0.0002, first BH threshold=5.34e-06. Increase inference.n_permutations when precise tail resolution is required.
  out["discrete"] = _finalize_discrete_table(
/users/xchen5/IsoVita/isovita.py:2794: RuntimeWarning: Continuous permutation resolution may be coarse for BH: minimum p=0.0002, first BH threshold=5.34e-06. Increase inference.n_permutations when precise tail resolution is required.
  out["continuous"] = _finalize_continuous_table(



IsoVita run complete
Output directory: /users/xchen5/IsoVita/data/HPC/isovita_results

Genes entering balance construction: 9,371
Successful balance features: 9,371
Discrete discoveries: 756
Continuous discoveries: 769
Unique genes in production-branch union: 1,009
